# 1단계: Claim-only 기준선 (Colab, GPU)

근거 없이 claim 문장만 보고 SUPPORTED / NOT_SUPPORTED를 맞힌다. 이 점수가 **하한선**이다. 이후 근거를 넣은 모델이 이보다 얼마나 나은지가 "근거의 효과"다.

| 셀 | 역할 |
|---|---|
| 1 | Drive 마운트, 0단계에서 저장한 jsonl 불러오기 |
| 2 | 다수 클래스 기준선. val과 test의 라벨 분포가 반대라서 (SUPPORTED 72% vs 46%) 두 셋에서 값이 다르다 |
| 3 | 토크나이즈. `make_text`가 모델 입력을 정한다. 2단계에서는 이 함수만 claim + 근거로 바꾼다 |
| 4 | 학습. `CLASS_WEIGHT=True`면 소수 클래스(NOT_SUPPORTED) 오답에 손실을 더 준다. 매 epoch val macro-F1로 최고 모델을 고른다 |
| 5 | test(공식 dev) 평가. accuracy와 macro-F1을 같이 본다 |

실험: `CLASS_WEIGHT`를 False / True로 한 번씩 돌려 비교한다.

In [7]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install -U transformers datasets accelerate

import pandas as pd
PROC = '/content/drive/MyDrive/fact-verification/data/processed'
LABELS = ['NOT_SUPPORTED', 'SUPPORTED']
dfs = {n: pd.read_json(f'{PROC}/hover2_{n}.jsonl', lines=True) for n in ['train', 'val', 'test']}
for d in dfs.values():
    d['labels'] = d.label.map(LABELS.index)
{n: len(d) for n, d in dfs.items()}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


{'train': 8166, 'val': 886, 'test': 1126}

In [8]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def scores(y, pred):
    return {'acc': accuracy_score(y, pred), 'macro_f1': f1_score(y, pred, average='macro')}

# train에서 많은 쪽(SUPPORTED)만 찍는 모델
for n in ['val', 'test']:
    y = dfs[n].labels
    print(n, 'majority(SUPPORTED)', scores(y, np.ones_like(y)))

val majority(SUPPORTED) {'acc': 0.7178329571106095, 'macro_f1': 0.4178712220762155}
test majority(SUPPORTED) {'acc': 0.4626998223801066, 'macro_f1': 0.31633272616879177}


In [9]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL = 'roberta-base'
tok = AutoTokenizer.from_pretrained(MODEL)

def make_text(row):
    return row['claim']

ds = {n: Dataset.from_pandas(d[['claim', 'labels']].reset_index(drop=True))
            .map(lambda r: tok(make_text(r), truncation=True, max_length=128))
      for n, d in dfs.items()}

Map:   0%|          | 0/8166 [00:00<?, ? examples/s]

Map:   0%|          | 0/886 [00:00<?, ? examples/s]

Map:   0%|          | 0/1126 [00:00<?, ? examples/s]

In [10]:
import torch
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding

CLASS_WEIGHT = True
counts = np.bincount(dfs['train'].labels)
weight = torch.tensor(len(dfs['train']) / (2 * counts), dtype=torch.float)  # 적은 클래스일수록 큰 가중치

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        out = model(**inputs)
        w = weight.to(out.logits.device) if CLASS_WEIGHT else None
        loss = torch.nn.functional.cross_entropy(out.logits, labels, weight=w)
        return (loss, out) if return_outputs else loss

def compute_metrics(p):
    return scores(p.label_ids, p.predictions.argmax(-1))

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
args = TrainingArguments(
    output_dir=f'/content/out/claim_only_cw{int(CLASS_WEIGHT)}',
    num_train_epochs=3, learning_rate=2e-5, per_device_train_batch_size=32, per_device_eval_batch_size=64,
    warmup_steps=0.1, weight_decay=0.01, fp16=True, seed=42,
    eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model='macro_f1', report_to='none',
)
trainer = WeightedTrainer(model=model, args=args, train_dataset=ds['train'], eval_dataset=ds['val'],
                          data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics)
trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Acc,Macro F1
1,No log,0.652556,0.494357,0.494354
2,0.663855,0.616590,0.632054,0.612299
3,0.663855,0.605522,0.696388,0.657367


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=768, training_loss=0.6432664394378662, metrics={'train_runtime': 137.2811, 'train_samples_per_second': 178.451, 'train_steps_per_second': 5.594, 'total_flos': 850998791902440.0, 'train_loss': 0.6432664394378662, 'epoch': 3.0})

In [11]:
from sklearn.metrics import confusion_matrix

for n in ['val', 'test']:
    p = trainer.predict(ds[n])
    pred = p.predictions.argmax(-1)
    print(n, scores(p.label_ids, pred))
    print(pd.DataFrame(confusion_matrix(p.label_ids, pred), index=[f'true_{l}' for l in LABELS], columns=[f'pred_{l}' for l in LABELS]))

val {'acc': 0.6963882618510158, 'macro_f1': 0.6573667148268748}
                    pred_NOT_SUPPORTED  pred_SUPPORTED
true_NOT_SUPPORTED                 159              91
true_SUPPORTED                     178             458


test {'acc': 0.6216696269982238, 'macro_f1': 0.6202847887832357}
                    pred_NOT_SUPPORTED  pred_SUPPORTED
true_NOT_SUPPORTED                 316             289
true_SUPPORTED                     137             384
